# Most Important Attention Mechanisms for ML/AI Interviews

## ⭐ TIER 1: MUST KNOW (You WILL be asked about these)

### **1. Scaled Dot-Product Attention**
**Why it matters**: Foundation of ALL transformers
- **Formula**: Attention(Q,K,V) = softmax(QK^T/√d_k)V
- **Interview Questions**:
  - "Explain how attention works"
  - "Why divide by √d_k?"
  - "What happens without the scaling?"
- **Why scaling matters**: 
  - Prevents attention scores from becoming too large
  - Keeps gradients stable during backprop
  - Without scaling: dot product grows with dimension d_k
- **Example**: 
  - If d_k=512, dot products can be ~512 in magnitude
  - After softmax, all gradients → 0 (vanishing gradient)
  - With √512≈22.6 scaling: scores stay ~0 to ~2.3 (stable)
- **When asked**: Explain simply: "We scale by √d_k to keep attention scores in reasonable range, preventing gradient issues"

**Complexity**: O(N²) where N is sequence length

---

### **2. Multi-Head Attention**
**Why it matters**: Every transformer uses this; BERT, GPT, all use multi-head
- **Concept**: Multiple "heads" (usually 8-12 or 16) running attention in parallel
- **Why multiple heads?**
  - Head 1 learns: "Find similar words"
  - Head 2 learns: "Track pronouns"
  - Head 3 learns: "Identify verb relationships"
  - Head 4 learns: "Learn syntax patterns"
  - Different heads capture different linguistic patterns
- **Formula**: MultiHead(Q,K,V) = Concat(head₁, ..., head_h)W^O
- **Interpretability**: You can visualize what each head attends to
- **Interview tip**: Explain with example "In 'The cat sat on the mat', one head attends to pronouns, another to word proximity"
- **Common question**: "Why use multiple heads instead of one big head?"
  - Answer: Better representation diversity, captures different aspects, empirically better

**Number of heads**: 
- BERT: 12 heads
- GPT-2: 12 heads
- GPT-3: 96 heads (wider = stronger)
- Llama: 32 heads

---

### **3. Self-Attention**
**Why it matters**: Core mechanism every LLM uses
- **Simple definition**: Token attends to all other tokens in **same sequence**
- **vs Cross-Attention**: 
  - Self: Same sequence (Word1 attends to Word2, Word3, Word4 in same sentence)
  - Cross: Different sequences (Decoder word attends to Encoder words in translation)
- **Interview questions**:
  - "Explain self-attention in one sentence"
  - "Why is self-attention better than RNNs?"
  - "What's the problem with self-attention?"
- **Benefits vs RNNs**:
  - RNN: Sequential, slow, hard to parallelize, forgets long context
  - Self-Attention: Parallel, fast, all tokens interact directly
- **Drawback**: O(N²) complexity (N = sequence length)
  - Sentence of 500 words = 250,000 attention computations
  - Sentence of 4000 tokens = 16 million attention computations

---

### **4. Causal/Masked Self-Attention**
**Why it matters**: Used in all generative LLMs (GPT, Llama, Mistral)
- **Key difference**: Can't attend to **future tokens**
- **Why?**: During generation, future tokens don't exist yet
  - Word 1 can see: Word 1
  - Word 2 can see: Word 1, 2 (NOT 3, 4, 5...)
  - Word 3 can see: Word 1, 2, 3 (NOT 4, 5...)
- **Masking mechanism**: Set future attention scores to -∞ before softmax
  - After softmax: -∞ becomes 0 (attending to nothing)
- **Interview scenario**: 
  - Q: "Why do LLMs use causal attention?"
  - A: "Because during generation, we predict word-by-word. Each word only knows previous words, not future ones that haven't been generated yet"
- **Attention matrix shape**: Lower triangular (autoregressive)

**Real example**:
```
Text: "The cat sat"
Position 1: Can attend to [1]
Position 2: Can attend to [1, 2]
Position 3: Can attend to [1, 2, 3]

Attention matrix:
[1, 0, 0]
[1, 1, 0]
[1, 1, 1]
```

---

### **5. Cross-Attention**
**Why it matters**: Used in encoder-decoder models (machine translation, image captioning, Stable Diffusion)
- **Definition**: Query from one sequence, Key/Value from **different sequence**
- **Real examples**:
  - Machine translation: English sentence (Q) ↔ French sentence (K,V)
  - Image captioning: Caption words (Q) attend to image features (K,V)
  - Stable Diffusion: Text prompt (Q) guides image generation (K,V)
- **Interview scenario**:
  - Q: "How does Stable Diffusion use attention?"
  - A: "Cross-attention: text prompt acts as queries, image features/noise as keys/values. Attention determines which parts of image to influence based on text"
- **vs Self-attention**: 
  - Self: A↔A
  - Cross: A↔B (asymmetric)

---

## ⭐⭐ TIER 2: VERY IMPORTANT (Asked in 60% of interviews)

### **6. Flash Attention** ⭐⭐⭐
**Why it matters**: PRACTICAL OPTIMIZATION everyone's using now (2024-2025)
- **Simple version**: Same exact attention, but computed **way faster**
- **Key insight**: Standard attention has GPU memory bottleneck
  - Loads Q, K, V from slow HBM (high bandwidth memory)
  - Computes attention
  - Writes back to HBM
  - This memory movement is the BOTTLENECK, not computation
- **Flash Attention**: Smart memory access pattern
  - Keeps small tiles of Q, K, V in fast SRAM
  - Computes locally
  - Only touches HBM once
- **Speed improvement**: 
  - BERT-large: 15-20% faster
  - GPT-2: 3× faster
  - Very long sequences: 7.6× faster
- **Memory**: 20× better memory usage
- **Quality**: **Exact same answer** (not approximate)
- **Interview angle**:
  - Q: "How do you speed up transformers for inference?"
  - A: "Flash Attention: IO-aware algorithm that rearranges memory access patterns. Exact same computation, just faster"
  - Q: "Isn't it approximate?"
  - A: "No! It's exact attention, just optimized memory layout"
- **Real-world**: Used in vLLM, production LLM inference
- **Variants**: Flash Attention v2 (even faster backward pass)

---

### **7. Rotary Position Embeddings (RoPE)** ⭐⭐⭐
**Why it matters**: Standard in modern LLMs (Llama, Falcon, Mistral, GPT-NeoX)
- **What it does**: Encodes position using **rotation matrices**
- **Simple idea**: 
  - Position n → Rotate query/key vectors by angle θ = n × base^(2i/d)
  - Different dimensions rotate at different speeds (like frequencies)
- **Why better than alternatives**:
  - Sinusoidal: Works poorly beyond training length (~512)
  - Learned embeddings: Can't generalize to longer sequences
  - RoPE: Better generalization, simpler math
- **Key advantage**: Naturally encodes **relative positions**
  - Attention between position i and j depends on their difference (i-j)
  - Not absolute position
- **Interview prep**:
  - Q: "Why do modern LLMs use RoPE instead of sinusoidal encoding?"
  - A: "RoPE encodes position as rotation, naturally captures relative positions, better numerical properties, faster training"
  - Q: "What's the limitation?"
  - A: "Limited extrapolation beyond training length. If trained on 512, struggles at 8K tokens"
- **Used in**: Llama 2/3, Falcon, Mistral, GPT-NeoX, PaLM
- **Variants**: NTK-aware RoPE (better extrapolation), YaRN

---

### **8. Attention with Linear Biases (ALiBi)** ⭐⭐⭐
**Why it matters**: Competing standard with RoPE; better for long sequences
- **Key difference from RoPE**: No learnable position embeddings at all
- **How it works**: Add bias term directly to attention scores based on distance
  - Attention_score = QK^T - m(i-j)
  - m = learnable slope (different per head)
  - Distance (i-j) tells how far apart tokens are
- **Why simpler than RoPE**:
  - RoPE: Rotates Q, K vectors (more complex)
  - ALiBi: Just adds a bias term (super simple)
- **Killer advantage**: Excellent length extrapolation
  - Train on 512 tokens → works on 8K+ tokens
  - Best extrapolation of any method
- **Speed**: ~30% faster training than RoPE (no rotation overhead)
- **Used in**: Falcon, MPT, newer efficient models
- **Interview scenario**:
  - Q: "Your model trained on 512 token limit, but needs 8K. What do you do?"
  - A: "Use ALiBi instead of RoPE. ALiBi's bias-based approach extrapolates much better. Train on 512, works on 8K+"
- **Trade-off**: Slightly lower quality than RoPE in some benchmarks, but much faster
- **Recommendation**: Use ALiBi if extrapolation is critical, RoPE if you want maximum quality

---

## ⭐⭐⭐ TIER 3: IMPORTANT FOR SCALE/EFFICIENCY (Asked in specialized roles)

### **9. Local/Window Attention (Longformer, BigBird)**
**Why it matters**: Only way to handle truly long documents (>8K tokens)
- **Problem**: Self-attention O(N²) is too slow for long sequences
  - 8K tokens = 64M attention operations
  - 32K tokens = 1B attention operations
- **Solution**: Only attend to nearby tokens (sliding window)
  - Token i attends to tokens in range [i-W/2, i+W/2]
  - Window size W: Typically 64, 128, 512
- **Complexity**: O(N·W) instead of O(N²)
  - Much faster for large N
  - If W << N: almost linear
- **Variant: Longformer**
  - Local attention + global attention
  - Global tokens ([CLS], task tokens) attend to everything
  - Local tokens only attend to window
  - Best of both worlds: local efficiency + global context
- **Used in**: Longformer (BERT variant), ETC, BigBird
- **Interview angle**:
  - Q: "How do you handle documents longer than 512 tokens?"
  - A: "Local window attention: each token only attends to nearby window of tokens, reducing O(N²) → O(N·W)"

---

### **10. Multi-Query Attention (MQA) & Grouped Query Attention (GQA)**
**Why it matters**: Inference speed optimization; crucial for production
- **Problem**: During generation, KV cache grows with sequence length
  - For 8K tokens: need to cache 8K key vectors + 8K value vectors
  - In batch of 32 requests: 32 × 8K × 2 = huge memory
- **MQA (Multi-Query Attention)**:
  - Multiple Q heads but **shared K, V heads**
  - Reduces KV cache by ~50%
  - Q: 12 heads, K: 1 head, V: 1 head
  - Faster generation (less KV to read)
- **GQA (Grouped Query Attention)** - Better than MQA
  - Q heads grouped: 12 Q heads → 3 groups
  - Each group shares one KV head
  - 4 Q heads per KV head
  - Balance: better quality than MQA, better speed than standard
- **Quality trade-off**: 
  - Standard: 100% quality baseline
  - GQA: ~99% quality, 2× faster inference
  - MQA: ~97% quality, 3× faster inference
- **Used in**: 
  - MQA: Palm-2, Falcon
  - GQA: Llama 2 (80B), Gemini, newer models
- **Interview scenario**:
  - Q: "How do you deploy Llama 2 70B for real-time chat?"
  - A: "GQA (Grouped Query Attention): shares KV heads across query groups. Reduces cache memory while maintaining quality"

---

### **11. Sparse Attention / BigBird**
**Why it matters**: Handles truly massive sequences efficiently
- **Problem**: Even local attention O(N·W) can be slow for N=1M tokens
- **BigBird solution**: Block-sparse attention with three components:
  1. **Local attention**: Window of nearby tokens
  2. **Global attention**: Selected tokens ([CLS], summary tokens) attend to all
  3. **Random attention**: Random token connections (improves connectivity)
- **Complexity**: O(N) with this combination
- **Extrapolation**: Can handle 8× longer sequences (4K → 32K)
- **Used in**: Long document classification, long sequence modeling
- **Interview angle**:
  - Q: "You need to process 32K token documents. Standard attention is too slow. What do you do?"
  - A: "BigBird: block-sparse attention combining local + global + random attention. O(N) complexity instead of O(N²)"

---

## ⭐⭐⭐⭐ TIER 4: VISION-SPECIFIC (If interviewing for vision/multimodal roles)

### **12. Spatial Attention (Vision)**
**Why it matters**: Computer vision understanding
- **Focus**: Which spatial regions/locations are important
- **Example**: In image of dog and cat, focus on dog region, ignore background
- **Simple form**: Assign weights to different spatial positions
- **Used in**: CBAM (Convolutional Block Attention Module), EfficientNet
- **Interview**: "How do you focus CNN on relevant regions?" → Spatial attention

---

### **13. Patch-Based Attention (Vision Transformer)**
**Why it matters**: Modern computer vision (replaces CNNs)
- **Key idea**: Vision Transformer (ViT) divides image into patches
  - 224×224 image → 14×14 = 196 patches of 16×16 pixels each
  - Each patch = one "token" (like word in NLP)
  - Apply standard transformer attention between patches
- **Advantage**: Pure attention, no convolutions
- **Trade-off**: Needs more data than CNN (for same performance)
- **Interview**: "How does Vision Transformer work?" → Patch-to-patch attention

---

### **14. Window-Based Attention (Swin Transformer)**
**Why it matters**: Efficient vision transformer
- **Idea**: ViT attention is O(N²) where N = number of patches
  - 14×14 = 196 patches → manageable
  - But for high-res images (28×28 = 784 patches) → slow
- **Solution: Swin Transformer**
  - Divide image into local windows (7×7)
  - Attention only within each window: O(N) total
  - Then shift windows so they overlap → connect nearby windows
- **Hierarchical**: Multi-scale pyramid (like ResNet)
- **Used in**: State-of-the-art vision models
- **Interview**: "How do you scale ViT to high-resolution images?" → Window attention

---

## 🎯 INTERVIEW CHEAT SHEET

### **If asked "Explain attention":**
1. Start: "Attention computes weighted sum of values based on similarity between query and key"
2. Formula: Attention(Q,K,V) = softmax(QK^T/√d_k)V
3. Why scaling: Prevents gradient issues
4. Multi-head: Different heads learn different patterns
5. Applications: Self-attention (transformers), Cross-attention (translation, captioning)

### **If asked "What's your favorite attention mechanism and why":**
**Good answer**: "RoPE for quality, ALiBi for long sequences. RoPE is standard in most modern LLMs (Llama, Falcon). ALiBi is better if you need to extrapolate beyond training length."

### **If asked "How do you optimize attention for production":**
**Good answer**: 
- Short sequences: Flash Attention (15-20% speedup, exact)
- Long sequences: ALiBi with window attention (extrapolation + efficiency)
- Inference: GQA to reduce KV cache memory

### **If asked "BERT vs GPT attention difference":**
**Answer**: 
- BERT: Bidirectional self-attention (can see all tokens)
- GPT: Causal self-attention (can only see previous tokens)

### **If asked "Why is attention better than RNN":**
**Answer**:
- RNN: Sequential, O(N) steps, vanishing gradients, slow to train
- Attention: Parallel, O(1) steps per layer, stable gradients, fast to train, captures long-range dependencies

---

## Summary: MUST KNOW for Interview

| Rank | Mechanism | Why Important | When Asked |
|------|-----------|---------------|-----------|
| 🔴 #1 | Scaled Dot-Product | Foundation of everything | Always |
| 🔴 #2 | Multi-Head Attention | Used everywhere | Always |
| 🔴 #3 | Self-Attention | Core of transformers | Always |
| 🔴 #4 | Causal Attention | All LLMs use this | Very likely |
| 🔴 #5 | Cross-Attention | Translation, captioning, Stable Diffusion | Likely |
| 🟡 #6 | Flash Attention | Speed optimization (modern) | 60% likely |
| 🟡 #7 | RoPE | Standard in Llama, Falcon, Mistral | 70% likely |
| 🟡 #8 | ALiBi | Long sequence extrapolation | 50% likely |
| 🟡 #9 | Local Window | Handling long documents | 40% likely |
| 🟡 #10 | GQA | Production inference optimization | 50% likely |
| 🟢 #11 | BigBird | Very long sequences | 30% likely |
| 🟢 #12+ | Vision/Patch/Window | Vision roles only | If vision role |

🔴 = Must know 100%
🟡 = Should know for top interviews
🟢 = Nice to know